# Laboratório 2: O Agente Gerador (RAG Básico)

**Objetivo deste script:** Criar o nosso primeiro funcionário digital. Este agente receberá uma pergunta do usuário, irá até o ChromaDB (a estante mágica que criamos no laboratório anterior) para buscar os trechos mais relevantes do manual e, em seguida, usará um modelo de linguagem (LLM) para formular uma resposta humana baseada *exclusivamente* no que encontrou.

É a fundação da arquitetura RAG (*Retrieval-Augmented Generation*).

In [1]:
# ==========================================
# CÉLULA 1: O AGENTE GERADOR E SALVAMENTO DE ESTADO
# ==========================================
import os
import warnings
warnings.filterwarnings("ignore")

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### Reconectando à Memória (VectorDB)
Antes de o agente falar, ele precisa saber como pesquisar. Vamos religar o modelo de tradução matemática (`all-MiniLM-L6-v2`) e apontar o caminho para a pasta onde salvamos o ChromaDB no script anterior.

In [2]:
# 1. Configuração de Caminhos Dinâmicos Seguros
DIRETORIO_ATUAL = os.getcwd()
RAIZ_DO_PROJETO = os.path.dirname(DIRETORIO_ATUAL)
DB_PATH = os.path.join(RAIZ_DO_PROJETO, "data", "vectordb")
LOG_PATH = os.path.join(RAIZ_DO_PROJETO, "data", "log_resposta_gerador.txt")

### Configurando o "Cérebro" e as Regras (Prompt)
Para garantir que o sistema rode de forma fluida e sem travar a máquina, delegamos o raciocínio para o modelo **Phi-3** via Ollama. Por ser incrivelmente otimizado, ele consegue fazer inferências sofisticadas operando confortavelmente dentro de restrições de hardware, como uma placa gráfica de 2GB de memória (GeForce 940MX).

O `PromptTemplate` é o manual de conduta do agente. É aqui que ordenamos que ele não invente respostas se não encontrar a informação.

In [3]:
# 2. Conectando a Memória e o Cérebro
print("🔌 Conectando ao Banco Vetorial...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
print("🧠 Acordando o Agente Gerador (Phi-3)...")
llm = OllamaLLM(model="phi3", temperature=0.1)

🔌 Conectando ao Banco Vetorial...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2854.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Acordando o Agente Gerador (Phi-3)...


### A Linha de Montagem (Pipeline LCEL)
O LangChain moderno usa a *LangChain Expression Language* (LCEL). É uma forma visual e profissional de dizer como os dados fluem, usando o símbolo `|` (pipe), que significa "passar o resultado para frente".

O fluxo é:
1. Pega a pergunta (`question`).
2. Usa a pergunta para buscar no banco (`context: retriever | formatar_docs`).
3. Junta a pergunta e os documentos e joga no `PROMPT` (A regra).
4. Envia o Prompt preenchido para o `llm` (O Cérebro).
5. Limpa a saída para um texto puro (`StrOutputParser`).

In [4]:
# 3. Prompt do Gerador
template_prompt = """Você é um programador RPA. 
Use APENAS os trechos de manuais técnicos fornecidos abaixo para responder.
Se a resposta não estiver no manual, diga "Não encontrei a informação no manual técnico fornecido." e não invente comandos.

Manual Técnico:
{context}

Pergunta: {question}

Resposta:"""

PROMPT = PromptTemplate.from_template(template_prompt)

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [5]:
# 4. Construindo a Esteira RAG
rag_chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

In [6]:
# 5. Execução
pergunta = "Como faço para dar um duplo clique na tela usando o PyAutoGUI?"
print(f"\n👤 Usuário: '{pergunta}'")
print("🤖 Processando RAG...")
resposta_final = rag_chain.invoke(pergunta)

print(f"\n--- Resposta Gerada ---\n{resposta_final}\n-----------------------\n")


👤 Usuário: 'Como faço para dar um duplo clique na tela usando o PyAutoGUI?'
🤖 Processando RAG...

--- Resposta Gerada ---
Para realizar um duplo clique em uma determinada posição da tela com o PyAutoGUI, você pode usar a função `doubleClick()`. Por exemplo:

```python
import pyautogui
pyautogui.doubleClick(x=100, y=200)  # Isso fará um duplo clique na posição (100, 200) da tela.
```

Lembre-se de substituir `x` e `y` pelos valores correspondentes à localização desejada no seu monitor para o clique do mouse ser executado corretamente.
-----------------------



In [7]:
# 6. O GRANDE DIFERENCIAL: Salvando o Estado
with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write(resposta_final)
    
print(f"💾 SUCESSO: A resposta foi salva fisicamente no arquivo de estado para auditoria futura.")

💾 SUCESSO: A resposta foi salva fisicamente no arquivo de estado para auditoria futura.
